# No modificar, todo funciona

In [1]:
# Librerias necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chisquare, pearsonr, ks_2samp
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import parametros as p
import os
import json
import pickle

In [2]:
# Nuevos datos con requerimientos de las llegadas
with open('../2. data/llegadas.pkl', 'rb') as file:
    llegadas_n = pickle.load(file)

In [3]:
# Paso el archivo de llegadas al mismo formato que usaba la funcion datos_llegadas
new_rows = []
requerimiento_dict = {1: "OR", 2: "ICU", 3: "SDU_WARD"}
"""
# nombre_unidades: nombre de las unidades
dict_unidades = {
    "OR": 1,
    "ICU": 2,
    "SDU/WARD": 3,
    "GA": 4,
    "ED": 5
}"""
# Paso el archivo de llegadas al mismo formato que usaba la funcion datos_llegadas
ciclos = len(llegadas_n["WL"])
for llegada in ["WL", "ED"]:
    for ciclo in range(1, ciclos + 1):
        for requerimiento in range(1, 4): # 1: "OR", 2: "ICU", 3: "SDU_WARD"
                    if llegada == "WL":
                        for grd in range(5,9):
                            for repeticiones in range(llegadas_n[llegada][ciclo][(grd, requerimiento)]):
                                new_row = {
                                    'MS_GRD': grd,
                                    'TI': ciclo * 12,
                                    'HOSPITAL': 0, # hospital cero no existe, pero es para poder iterar sin errores
                                    'LLEGADA': llegada,
                                    'REQUERIMIENTO': requerimiento_dict[requerimiento]
                                }
                                new_rows.append(new_row)
                    elif llegada == "ED":
                        for hospital in range(1, 4):
                            for grd in range(1,5):
                                for repeticiones in range(llegadas_n[llegada][ciclo][(hospital, grd, requerimiento)]):
                                    new_row = {
                                        'MS_GRD': grd,
                                        'TI': ciclo * 12,
                                        'HOSPITAL': hospital,
                                        'LLEGADA': llegada,
                                        'REQUERIMIENTO': requerimiento_dict[requerimiento]
                                    }
                                    new_rows.append(new_row)

# Crear un nuevo DataFrame con las nuevas filas
new_llegadas = pd.DataFrame(new_rows)
                    

In [4]:

# Defino la funcion para el ajuste de los datos (la modifico para que incluya hospitales)
def datos_llegadas(llegadas, requerimiento, grd, hospital):
    # Para visualizar los datos de llegadas (cambiar requerimiento y grd que se quiera ver)
    v1 = llegadas[(llegadas["REQUERIMIENTO"] == requerimiento) & (llegadas["MS_GRD"] == grd) & (llegadas["HOSPITAL"] == hospital)]
    vector = v1["TI"].value_counts().reset_index().sort_values(by="TI")
    vector.columns = ["TI", "OCURRENCIAS"]
    v2 = vector["OCURRENCIAS"].value_counts().reset_index()
    v2.columns = ["VALOR", "OCURRENCIAS"]
    v2.sort_values(by="VALOR", inplace=True)

    # Agregar la cantidad de veces que no llego nadie, osea valor 0
    ciclos_simulacion = int(llegadas["TI"].max()/12)
    # agregar nueva fila
    cero_ocurrencias = {"VALOR": 0, "OCURRENCIAS": ciclos_simulacion - v2["OCURRENCIAS"].sum()}
    v2 = pd.concat([v2, pd.DataFrame([cero_ocurrencias])], ignore_index=True)
    # Ordenar por VALOR
    v2.sort_values(by="VALOR", inplace=True)
    v2.reset_index(inplace=True)
    v2.drop(columns=["index"], inplace=True)
    return v2



In [5]:
resultados = {}
for hospital in range(0, 4): # El cero es para la gente en WL
    resultados[hospital] = {}
    for requerimiento in ["OR", "ICU", "SDU_WARD"]:
            id_requerimiento = p.dict_unidades[requerimiento.replace("_","/")]
            resultados[hospital][id_requerimiento] = {}
            for grd in range(1, 9):
                resultados[hospital][id_requerimiento][grd] = {}
                v2 = datos_llegadas(new_llegadas, requerimiento, grd, hospital)
                if len(v2) > 1: # Mayor a 1 porque a todos se les agrega el cero
                    #best_h, final_kde_pmf, chi2, p_val = ajuste_kde_triangular(v2, requerimiento, grd, plot, hospital, a1=1, a2=1, initial_h=0.3, p_lb=0.065, p_ub=0.1)
                    if True is not None:
                        vector = v2.copy()
                        # Repito este codigo para calcular las metricas
                        values = np.arange(int(min(vector["VALOR"])), int(max(vector["VALOR"])) + 1) # Soporte para el KDE
                        vector_filled = vector.set_index("VALOR").reindex(values, fill_value=1).reset_index()
                        empirical_counts = vector_filled["OCURRENCIAS"].values
                        los_samples = np.repeat(vector_filled["VALOR"], vector_filled["OCURRENCIAS"])
                        empirical_pmf = empirical_counts / empirical_counts.sum()
                        resultados[hospital][id_requerimiento][grd] = {
                        "final_kde_pmf": empirical_pmf.tolist(),
                        }

In [6]:
def save_dict_as_json(data_dict, filename, folder):
    os.makedirs(folder, exist_ok=True)  # Create folder if it doesn't exist
    path = os.path.join(folder, filename)
    with open(path, 'w') as f:
        json.dump(data_dict, f, indent=4)
    print(f"Dictionary saved to: {path}")

# Save the results to a JSON file
save_dict_as_json(resultados, filename="llegadas_empiricas.json", folder="resultados incertidumbre")

Dictionary saved to: resultados incertidumbre/llegadas_empiricas.json
